# Summarize

数据概览。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import sys
sys.path.insert(0, str(Path('../../modules').resolve()))
from pipeline_config import *

In [ ]:
#  设置中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号

# 路径从配置读取
user_file = ORIGINALITY_FILE
ai_file = AI_EVENTS_FILE

user_msgs = pd.read_csv(user_file, encoding='utf-8')
ai_events = pd.read_pickle(ai_file)

# 查看数据基本信息
print(f"用户消息总数: {len(user_msgs)}")
print(f"AI 调用事件总数: {len(ai_events)}")
print(f"参与被试数: {user_msgs['participant_id'].nunique()}")

In [ ]:
# 按被试和轮次统计想法总数
thoughts_per_trial_participant = user_msgs.groupby(['participant_id', 'trial_index']).size().reset_index(name='total_thoughts')

print("单轮想法总数描述统计：")
print(thoughts_per_trial_participant['total_thoughts'].describe())

# 按被试和轮次统计 AI 调用次数
calls_per_trial_participant = ai_events.groupby(['participant_id', 'trial_index']).size().reset_index(name='total_calls')

print("\n单轮 AI 调用次数描述统计：")
print(calls_per_trial_participant['total_calls'].describe())

# 合并到一个图中，横向排布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))  # 创建左右两个子图，总尺寸加大

# 左边：想法总数分布
sns.histplot(thoughts_per_trial_participant['total_thoughts'], bins=20, kde=True, ax=ax1)
ax1.set_xlabel('想法总数', fontsize=14)
ax1.set_ylabel('试次数', fontsize=14)
ax1.set_title('想法总数分布', fontsize=16)

# 右边：AI调用次数分布
sns.histplot(calls_per_trial_participant['total_calls'], bins=20, kde=True, ax=ax2)
ax2.set_xlabel('AI 调用次数', fontsize=14)
ax2.set_ylabel('试次数', fontsize=14)
ax2.set_title('AI 调用次数分布', fontsize=16)

sns.despine() # 去掉图的顶部和右侧边框
plt.tight_layout()  # 自动调整子图间距
plt.show()

In [ ]:
# 所有被试 ID（来自用户消息，确保包含所有参与的被试）
all_participants = user_msgs['participant_id'].unique()
calling_participants = ai_events['participant_id'].unique()
zero_callers = set(all_participants) - set(calling_participants)
zero_count = len(zero_callers)
total_participants = len(all_participants)
zero_ratio = zero_count / total_participants

print(f"\n总被试数: {total_participants}")
print(f"无调用者人数: {zero_count}")
print(f"无调用者比例: {zero_ratio:.2%}")

# 也可以绘制饼图
plt.figure(figsize=(6,6))
plt.pie([zero_count, total_participants - zero_count], labels=['无调用', '有调用'], autopct='%1.1f%%', startangle=90, colors=['#ff9999','#66b3ff'])
plt.title('无调用者比例')
plt.show()

In [ ]:
# 按轮次统计平均想法数
thoughts_per_trial = user_msgs.groupby('trial_index').size().reset_index(name='count')
thoughts_per_trial['trial_index'] = thoughts_per_trial['trial_index'].astype(int)
print("\n各轮次想法总数：")
print(thoughts_per_trial)

# 按轮次统计平均调用数
calls_per_trial = ai_events.groupby('trial_index').size().reset_index(name='count')
calls_per_trial['trial_index'] = calls_per_trial['trial_index'].astype(int)
print("\n各轮次 AI 调用总数：")
print(calls_per_trial)

# 全局字体设置
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

# 绘制柱状图
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
sns.barplot(data=thoughts_per_trial, x='trial_index', y='count', ax=axes[0],
            palette='Blues_d', width=0.45)
axes[0].set_title('各轮次想法总数', fontsize=14, fontweight='bold')
axes[0].set_xlabel('轮次', fontsize=12)
axes[0].set_ylabel('想法总数', fontsize=12)

sns.barplot(data=calls_per_trial, x='trial_index', y='count', ax=axes[1],
            palette='Oranges_d', width=0.45)
axes[1].set_title('各轮次 AI 调用总数', fontsize=14, fontweight='bold')
axes[1].set_xlabel('轮次', fontsize=12)
axes[1].set_ylabel('调用总数', fontsize=12)

# 在柱子上标注数值 + 拓展y轴上限防止数字溢出
for ax in axes:
    ymax = 0
    for bar in ax.patches:
        height = bar.get_height()
        ymax = max(ymax, height)
        ax.text(bar.get_x() + bar.get_width() / 2., height + 2,
                f'{int(height)}', ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, ymax * 1.15)

plt.tight_layout(pad=1)

# 保存高清图到桌面
save_path = r'C:\Users\lenovo\Desktop\各轮次想法与AI调用统计.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n图片已保存至: {save_path}")
plt.show()

## 物品分布

检查正式任务中各实验组物品的随机分配是否均匀，确保实验设计平衡。

In [ ]:
# 加载原始回答数据检查物品分配
responses = pd.read_csv(RESPONSES_FILE)

# 过滤练习轮次（trial_index=0），去除同一被试对同一物品的重复记录
item_count = responses[responses['trial_index'] != 0].drop_duplicates(
    subset=['participant_id', 'item_name']
)['item_name'].value_counts()